In [1]:
import yaml

In [2]:
from anndata import read_h5ad

In [3]:
from os.path import join

In [4]:
sample_group_pairs = [
    # AKI vs. HRT
    ('EnrollmentCategory', ('Healthy Reference', 'AKI')),
    # AKI vs. H-CKD. (H-CKD not in enrollment category values anymore. Should I use "Hypertension History" Yes/No column?)
    ('EnrollmentCategory', ('AKI', 'CKD')),
    # D-CKD vs. HRT. (D-CKD not in enrollment category values anymore. Should I use "Diabetes History" Yes/No column?)
    ('EnrollmentCategory', ('CKD', 'Healthy Reference')),
    # Diabetes CKD vs. Hypertension CKD. (DKD nor H-CKD not in enrollment category values anymore. Should I use Yes/No columns?)
    #('EnrollmentCategory', ('DKD', 'H-CKD')),
    # D-CKD vs. HRT
    ('AdjudicatedCategory', ('Diabetic Kidney Disease', 'Healthy Reference')),
    # Acute tubular injury vs. HRT
    ('AdjudicatedCategory', ('Acute Tubular Injury', 'Healthy Reference')),
    # Acute interstitial nephritis vs. HRT
    ('AdjudicatedCategory', ('Acute Interstitial Nephritis', 'Healthy Reference')),
    # Diabetes CKD vs. Hypertension CKD
    ('AdjudicatedCategory', ('Diabetic Kidney Disease', 'Hypertensive Kidney Disease')),
    # ATN vs. AIN
    ('AdjudicatedCategory', ('Acute Interstitial Nephritis', 'Acute Tubular Injury')),

    # TODO: use Diabetes History and Hypertension History columns here.
]
cell_type_cols = [
    "subclass_l3",
    "subclass_l2",
    "subclass_l1",
]

In [5]:
adata_path = join("data", "raw", "kpmp-aug-2025", "SingleNucleus_KPMP_Explorer_05182025.h5ad")
clinical_path = join("data", "raw", "kpmp-aug-2025", "20250606_OpenAccessClinicalData.csv")

In [6]:
adata = read_h5ad(adata_path)

In [7]:
cell_types = {}
for colname in cell_type_cols:
    orig_colname = colname.replace("_", ".")
    cell_types[colname] = sorted(adata.obs[orig_colname].unique().tolist(), key=lambda v: v.lower())

In [8]:
sample_group_pairs_dict = [
    { "colname": t[0], "lhs": t[1][0], "rhs": t[1][1] }
    for t in sample_group_pairs
]

In [9]:
#yaml_output = yaml.dump({ "cell_types": cell_types, "sample_group_pairs": sample_group_pairs_dict }, default_flow_style=False)
#print(yaml_output)

In [10]:
import numpy as np
should_subset = True
if should_subset:
    print("SUBSETTING")
    # subset using random sample so that multiple sample groups are represented to enable comparison
    np.random.seed(1)
    obs_subset = np.random.choice(adata.obs.index.tolist(), size=20_000, replace=False).tolist()
    var_slice = slice(None)
    adata = adata[obs_subset, var_slice].copy()

SUBSETTING


In [11]:
# CLEANUP FROM SCRIPT
import pandas as pd
# Join adata.obs with clinical data from CSV
clinical_data = pd.read_csv(clinical_path)

adata.obs = adata.obs.merge(clinical_data, left_on="patient", right_on="Participant ID", how="left")

# We could have done a left join, but then we would have to filter out samples that do not have clinical data later.
# We also do not want strings to be converted to NaN, as these cause Zarr writing errors like "TypeError: expected unicode string, found nan".

# This effectively does an inner join. We cannot use how="inner", since this would only affect adata.obs, and not other anndata fields.
has_clinical_data = ~adata.obs["Participant ID"].isna()
adata = adata[has_clinical_data, :].copy()

print(adata.obs.head())

adata.obs["Primary Adjudicated Category"] = adata.obs["Primary Adjudicated Category"].fillna("NA")

# Cleanup of sample-level data
def clean_adjudicated_category(row):
    if row["Primary Adjudicated Category"] != "NA":
        return row["Primary Adjudicated Category"]
    else:
        # The row was empty, so perhaps this sample has not yet been adjudicated.
        # However, we also need to check that this was not a "Healthy Reference" sample,
        # as these never go through the adjudication process.
        if row["Enrollment Category"] in ["Healthy Reference"]:
            return "Healthy Reference"
        return ""
adata.obs["AdjudicatedCategory"] = adata.obs.apply(clean_adjudicated_category, axis='columns')
adata.obs["EnrollmentCategory"] = adata.obs["Enrollment Category"]

# TODO: process other clinical columns? Sex, age group, etc.

adata.obs = adata.obs.rename(columns={"subclass.l1": "subclass_l1", "subclass.l2": "subclass_l2", "subclass.l3": "subclass_l3"})

for colname in adata.obs.columns:
    if pd.api.types.is_string_dtype(adata.obs[colname]) or str(adata.obs[colname].dtype) == "object":
        print(f"Filling NAs in string column {colname} with 'NA'")
        adata.obs[colname] = adata.obs[colname].fillna("NA")
    else:
        print(f"Not filling NAs in non-string column {colname} of type {adata.obs[colname].dtype}")

# Column names cannot contain slashes
adata.obs = adata.obs.rename(columns=dict(zip(adata.obs.columns, [c.replace("/", " per ") for c in adata.obs.columns])))


/Users/mkeller/.local/share/uv/python/cpython-3.11.9-macos-aarch64-none/lib/python3.11/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


  library_id  nCount_RNA  nFeature_RNA  percent.er  percent.mt  \
0      KB146      4638.0        2531.0    1.209934    1.549565   
1       KB72      3563.0        2092.0    2.432575    5.790587   
2       KB49      5434.0        2582.0    0.091743    0.293578   
3       KB24       623.0         525.0    4.960000    0.320000   
4       KB78      4445.0        2678.0    4.881626    6.871988   

    experiment_id             specimen   patient region  percent.cortex  ...  \
0  KPMP_20231116C  S-2102-003511_D1_N1  34-10393      C              95  ...   
1  KPMP_20220606A  S-2108-005585_D2_N1    164-10      C              90  ...   
2  KPMP_20211013B     S-2107-011583_R1     164-6      C             100  ...   
3  KPMP_20201209D     S-2001-000149_R1  29-10016      C              85  ...   
4  KPMP_20220908E  S-2108-021709_D1_N1  28-12372      C             100  ...   

   Baseline eGFR (ml/min/1.73m2) Baseline eGFR (ml/min/1.73m2) (Binned)  \
0                      82.914527               

In [12]:
adata.layers['counts']

<18694x36588 sparse matrix of type '<class 'numpy.float64'>'
	with 42817000 stored elements in Compressed Sparse Column format>

In [13]:
import decoupler as dc

/Users/mkeller/research/dbmi/vitessce/compasce/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
# References:
# - https://pertpy.readthedocs.io/en/stable/tutorials/notebooks/differential_gene_expression.html#pseudobulks
# - https://decoupler.readthedocs.io/en/latest/api/generated/decoupler.pp.pseudobulk.html

In [15]:
pdata = dc.pp.pseudobulk(adata, sample_col="specimen", groups_col="subclass_l1", layer="counts", mode="sum", verbose=True)

2025-11-10 15:25:15 | [INFO] Extracted omics mat with 18694 rows (observations) and 36588 columns (features)
2025-11-10 15:25:16 | [INFO] Generating 2916 profiles: 162 samples x 18 groups
2025-11-10 15:25:16 | [INFO] Using function sum to aggregate observations
2025-11-10 15:25:16 | [INFO] group=ATL	sample=18-142-3-M2	cells=0	counts=0.0
2025-11-10 15:25:16 | [INFO] group=ATL	sample=18-162-2-M2	cells=0	counts=0.0
2025-11-10 15:25:16 | [INFO] group=ATL	sample=18-312-2-M2	cells=0	counts=0.0
2025-11-10 15:25:16 | [INFO] group=ATL	sample=446_B1	cells=1	counts=2037.0
2025-11-10 15:25:16 | [INFO] group=ATL	sample=446_B3	cells=0	counts=0.0
2025-11-10 15:25:16 | [INFO] group=ATL	sample=K1800364_1	cells=0	counts=0.0
2025-11-10 15:25:16 | [INFO] group=ATL	sample=K1800430_3_R3	cells=0	counts=0.0
2025-11-10 15:25:16 | [INFO] group=ATL	sample=K1900019_1_R3	cells=5	counts=50800.0
2025-11-10 15:25:16 | [INFO] group=ATL	sample=K1900174_2	cells=0	counts=0.0
2025-11-10 15:25:16 | [INFO] group=ATL	sample=

In [16]:
pdata

AnnData object with n_obs × n_vars = 2916 × 36588
    obs: 'specimen', 'subclass_l1', 'patient', 'region', 'percent.cortex', 'percent.medulla', 'class', 'Participant ID', 'Tissue Source', 'Protocol', 'Sample Type', 'Enrollment Category', 'Primary Adjudicated Category', 'Sex', 'Age (Years) (Binned)', 'Race', 'KDIGO Stage', 'Baseline eGFR (ml per min per 1.73m2)', 'Baseline eGFR (ml per min per 1.73m2) (Binned)', 'Proteinuria (mg) (Binned)', 'A1c (%) (Binned)', 'Albuminuria (mg) (Binned)', 'Diabetes History', 'Diabetes Duration (Years)', 'Hypertension History', 'Hypertension Duration (Years)', 'On RAAS Blockade', 'AdjudicatedCategory', 'EnrollmentCategory', 'psbulk_cells', 'psbulk_counts'
    var: 'gene_symbol', 'ensembl_id'
    layers: 'psbulk_props'

In [17]:
import pertpy as pt

/Users/mkeller/research/dbmi/vitessce/compasce/.venv/lib/python3.11/site-packages/anndata/__init__.py:44: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  return module_get_attr_redirect(attr_name, deprecated_mapping=_DEPRECATED)
/Users/mkeller/research/dbmi/vitessce/compasce/.venv/lib/python3.11/site-packages/anndata/__init__.py:44: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  return module_get_attr_redirect(attr_name, deprecated_mapping=_DEPRECATED)
/Users/mkeller/research/dbmi/vitessce/compasce/.venv/lib/python3.11/site-packages/anndata/__init__.py:44: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  return module_get_attr_redirect(attr_name, deprecated_mapping=_DEPRECATED)
/Users/mkeller/research/dbmi/vitessce/compasce/.venv/lib/python3.11/site-packages/anndata/experimental/__init__.py:48: FutureWarning: Importing CSC

In [20]:
pds2 = pt.tl.PyDESeq2(adata=pdata, design=f"~EnrollmentCategory")
pds2.fit()

Using None as control genes, passed at DeseqDataSet initialization


Fitting size factors...
/Users/mkeller/research/dbmi/vitessce/compasce/.venv/lib/python3.11/site-packages/pydeseq2/dds.py:532: UserWarning: Every gene contains at least one zero, cannot compute log geometric means. Switching to iterative mode.
  self.fit_size_factors(


IndexError: too many indices for array: array is 1-dimensional, but 2 were indexed

In [34]:
from importlib.metadata import version
version('pydeseq2')

'0.5.0'

In [32]:
pdata.varm["non_zero"] = ~(pdata.X == 0).all(axis=0)

In [33]:
pdata.varm["non_zero"]

array([[ True],
       [ True],
       [ True],
       ...,
       [ True],
       [ True],
       [ True]])

In [31]:
~(pdata.X == 0).all(axis=0)

array([ True,  True,  True, ...,  True,  True,  True])

In [24]:
pdata.n_vars

36588

In [25]:
pdata.varm["non_zero"]

array([[ True],
       [ True],
       [ True],
       ...,
       [ True],
       [ True],
       [ True]])

In [29]:
np.arange(pdata.n_vars)[pdata.varm["non_zero"]]

IndexError: too many indices for array: array is 1-dimensional, but 2 were indexed

In [ ]:
print("test")